# Лекция: Описательные статистики в Python

**Дисциплина:** Введение в анализ больших данных  
**Задание 3** (адаптация с языка R на Python)

В оригинальном задании используется датасет **NMES1988** (National Medical Expenditure Survey 1988) — 4406 наблюдений, 19 переменных.

Мы работаем с переменными:
- `visits` — количество посещений врача
- `health` — самооценка здоровья (фактор)
- `chronic` — число хронических заболеваний
- `adl` — ограничение повседневной деятельности
- `region` — регион проживания
- `age` — возраст (в десятках лет)
- `gender` — пол
- `married` — семейное положение
- `school` — годы обучения
- `income` — доход (в 10 000 $)
- `employed`, `insurance` — факторы

В Python основные инструменты описательной статистики:
- **pandas** — `describe()`, `groupby()`, `median()`, `quantile()`
- **numpy** / **scipy.stats** — среднее, ст. отклонение, асимметрия, эксцесс
- **matplotlib** / **seaborn** — гистограммы, boxplot, barplot


## 0. Импорт библиотек и загрузка данных


In [ ]:
# !pip install numpy pandas scipy matplotlib seaborn

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")

print("Библиотеки загружены")


In [ ]:
# Загрузка датасета NMES1988
# Источник: https://vincentarelbundock.github.io/Rdatasets/csv/AER/NMES1988.csv

url = "https://vincentarelbundock.github.io/Rdatasets/csv/AER/NMES1988.csv"
df = pd.read_csv(url)

print("Размерность таблицы:", df.shape)
print("\nСтолбцы:")
print(df.columns.tolist())
print("\nПервые 5 строк:")
df.head()


In [ ]:
# Выберем нужные переменные (как в задании)
cols = ["visits", "health", "chronic", "adl", "region", "age",
        "gender", "married", "school", "income", "employed", "insurance"]

# Проверяем, какие из них есть
available = [c for c in cols if c in df.columns]
print("Доступные переменные:", available)

nmes = df[available].copy()
print("\nРазмерность рабочей таблицы:", nmes.shape)
nmes.info()


### Краткий обзор данных


In [ ]:
nmes.describe(include="all")


---
## 1. Гистограмма показателя `visits` с кривой плотности

В R:
```r
hist(X, freq = FALSE, ...)
lines(density(X), ...)
```

В Python:
- `plt.hist(..., density=True)` + теоретическая или KDE-кривая
- или `sns.histplot(..., kde=True)`


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Гистограмма (density=True — нормированная)
ax.hist(nmes["visits"], bins=40, density=True, color="lightblue",
        edgecolor="black", alpha=0.7, label="Гистограмма")

# Ядерная оценка плотности (аналог density() в R)
from scipy.stats import gaussian_kde
kde = gaussian_kde(nmes["visits"].dropna())
x_grid = np.linspace(0, nmes["visits"].max(), 300)
ax.plot(x_grid, kde(x_grid), color="red", linewidth=2, label="KDE (плотность)")

ax.set_xlabel("Количество посещений врача (visits)")
ax.set_ylabel("Плотность вероятности")
ax.set_title("Гистограмма показателя visits с кривой плотности")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Более простой вариант через seaborn
plt.figure(figsize=(10, 6))
sns.histplot(nmes["visits"], bins=40, kde=True, color="steelblue",
             edgecolor="black", alpha=0.7)
plt.xlabel("visits")
plt.ylabel("Плотность / Частота")
plt.title("Гистограмма visits + KDE (seaborn)")
plt.tight_layout()
plt.show()


---
## 2. Среднее и стандартная ошибка среднего (SE)

В R:
```r
mean(x)
SE = sd(x) / sqrt(length(x))
```

Стандартная ошибка среднего показывает точность оценки среднего значения.


In [ ]:
visits = nmes["visits"].dropna()

mean_visits = visits.mean()
sd_visits = visits.std(ddof=1)          # выборочное ст. отклонение
n = len(visits)
se_visits = sd_visits / np.sqrt(n)

print(f"Среднее значение visits     : {mean_visits:.4f}")
print(f"Ст. отклонение (sd)         : {sd_visits:.4f}")
print(f"Объём выборки (n)           : {n}")
print(f"Стандартная ошибка среднего : {se_visits:.4f}")


---
## 3. Медиана и квартили

В R: `median()`, `quantile()`, `summary()`

В pandas: `.median()`, `.quantile()`, `.describe()`


In [ ]:
print("Медиана visits:", visits.median())

print("\nКвартили (0%, 25%, 50%, 75%, 100%):")
print(visits.quantile([0, 0.25, 0.5, 0.75, 1.0]))

print("\nПолный summary (describe):")
print(visits.describe())


---
## 4. Коэффициент асимметрии (skewness) и эксцесса (kurtosis)

В R (пакет moments): `skewness()`, `kurtosis()`

В Python:
- `scipy.stats.skew()`
- `scipy.stats.kurtosis()` (по умолчанию excess kurtosis, т.е. нормальное = 0)
- или `pandas.Series.skew()`, `.kurtosis()`


In [ ]:
skewness = stats.skew(visits)
kurtosis = stats.kurtosis(visits)   # excess kurtosis (нормальное ≈ 0)

print(f"Асимметрия (skewness) : {skewness:.4f}")
print(f"Эксцесс (kurtosis)    : {kurtosis:.4f}")

# Интерпретация:
# Асимметрия > 0 → правосторонняя (длинный правый хвост) — типично для счётных данных
# Эксцесс > 0    → более островершинное распределение, чем нормальное


---
## 5. Медианы и квартили количественных переменных отдельно для мужчин и женщин

В R: `tapply(x, INDEX = gender, FUN = summary)`

В Python: `groupby("gender")` + `.describe()` / `.median()`


In [ ]:
# Сначала посмотрим уникальные значения gender
print("Уровни gender:", nmes["gender"].unique())

# Описательные статистики visits по полу
print("\n=== visits по полу ===")
print(nmes.groupby("gender")["visits"].describe())


In [ ]:
# Можно сразу несколько количественных переменных
quant_cols = ["visits", "chronic", "age", "school", "income"]

print("Медианы по полу:")
print(nmes.groupby("gender")[quant_cols].median())

print("\nКвартили (25% и 75%) visits по полу:")
print(nmes.groupby("gender")["visits"].quantile([0.25, 0.5, 0.75]))


---
## 6. Диаграмма размаха (boxplot) количества посещений по полу

В R: `boxplot(visits ~ gender, data = nmes)`

В Python: `sns.boxplot()` или `plt.boxplot()`


In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(data=nmes, x="gender", y="visits", palette="Set2")
plt.xlabel("Пол (gender)")
plt.ylabel("Количество посещений (visits)")
plt.title("Диаграмма размаха: visits по полу")
plt.tight_layout()
plt.show()


**Как читать boxplot:**
- Линия внутри «ящика» — медиана
- Границы ящика — 1-й и 3-й квартили (Q1, Q3)
- Усы — обычно до 1.5 × IQR
- Точки за усами — потенциальные выбросы


---
## 7. Медианные значения visits по gender и region одновременно

В R: `tapply(x, INDEX = list(gender, region), FUN = median)`

В Python: `groupby(["gender", "region"])["visits"].median()`


In [ ]:
medians = nmes.groupby(["gender", "region"])["visits"].median().unstack()
print("Медианы visits по полу и региону:")
print(medians)


---
## 8. Столбиковые диаграммы на основе медиан visits по gender и region

В R: `barplot(Means, ...)`


In [ ]:
# Медианы по полу
med_gender = nmes.groupby("gender")["visits"].median()

plt.figure(figsize=(7, 5))
med_gender.plot(kind="bar", color="steelblue", edgecolor="black")
plt.ylabel("Медиана visits")
plt.xlabel("Пол")
plt.title("Медианное число посещений врача по полу")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Медианы по региону
med_region = nmes.groupby("region")["visits"].median().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
med_region.plot(kind="bar", color="darkorange", edgecolor="black")
plt.ylabel("Медиана visits")
plt.xlabel("Регион")
plt.title("Медианное число посещений врача по регионам")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Сгруппированная столбчатая диаграмма: gender × region
medians_plot = nmes.groupby(["region", "gender"])["visits"].median().unstack()

medians_plot.plot(kind="bar", figsize=(10, 6), edgecolor="black")
plt.ylabel("Медиана visits")
plt.xlabel("Регион")
plt.title("Медианное число посещений по региону и полу")
plt.xticks(rotation=45)
plt.legend(title="Пол")
plt.tight_layout()
plt.show()


---
## 9. Медианный возраст по категориям married (вся выборка и по полу)

Задание: рассчитать медианный возраст по `married` для всей выборки и отдельно для мужчин и женщин, визуализировать `barplot`.


In [ ]:
# Вся выборка
med_age_married = nmes.groupby("married")["age"].median()
print("Медианный возраст по семейному положению (вся выборка):")
print(med_age_married)

# По полу
med_age_gender_married = nmes.groupby(["gender", "married"])["age"].median().unstack()
print("\nМедианный возраст по полу и семейному положению:")
print(med_age_gender_married)


In [ ]:
# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Вся выборка
med_age_married.plot(kind="bar", ax=axes[0], color="teal", edgecolor="black")
axes[0].set_title("Медианный возраст по married (вся выборка)")
axes[0].set_ylabel("Медиана age (в десятках лет)")
axes[0].set_xlabel("Married")
axes[0].tick_params(axis='x', rotation=0)

# По полу
med_age_gender_married.plot(kind="bar", ax=axes[1], edgecolor="black")
axes[1].set_title("Медианный возраст по married и gender")
axes[1].set_ylabel("Медиана age")
axes[1].set_xlabel("Пол")
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title="Married")

plt.tight_layout()
plt.show()


### Краткий анализ
- Обратите внимание на различия медианного возраста между женатыми/замужними и неженатыми.
- Сравните паттерны у мужчин и женщин — часто возраст вступления в брак и структура выборки отличаются.


---
## Шпаргалка: R → Python (описательные статистики)

| Задача в R | Python (pandas / scipy / seaborn) |
|------------|-----------------------------------|
| `read.csv(...)` | `pd.read_csv(url)` |
| `mean(x)` | `x.mean()` / `np.mean(x)` |
| `sd(x)` | `x.std(ddof=1)` |
| `SE = sd(x)/sqrt(n)` | `x.std(ddof=1) / np.sqrt(len(x))` |
| `median(x)` | `x.median()` |
| `quantile(x)` / `summary(x)` | `x.quantile()` / `x.describe()` |
| `skewness(x)` | `stats.skew(x)` / `x.skew()` |
| `kurtosis(x)` | `stats.kurtosis(x)` / `x.kurtosis()` |
| `tapply(x, INDEX, FUN)` | `df.groupby(...)[col].agg(...)` |
| `boxplot(y ~ x)` | `sns.boxplot(x=..., y=...)` |
| `barplot(Means)` | `series.plot(kind="bar")` / `sns.barplot()` |
| `hist(..., freq=FALSE)` + `lines(density())` | `plt.hist(..., density=True)` + KDE / `sns.histplot(..., kde=True)` |

---
## Рекомендации

1. Всегда проверяйте типы данных (`df.info()`, `df.dtypes`).
2. Для категориальных переменных удобно использовать `.value_counts()` и `pd.crosstab()`.
3. При работе с выбросами boxplot — один из лучших инструментов визуального анализа.
4. Документация: [pandas](https://pandas.pydata.org/docs/), [seaborn](https://seaborn.pydata.org/).

**Удачи с выполнением Задания 3!**
